In [ ]:
from google.colab import drive
import gzip
import random
import networkx as nx
from collections import deque
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

drive.mount('/content/drive')

%cd /content/drive/MyDrive/Colab Notebooks/AM

In [ ]:
adj = {}

with gzip.open('web-Google.txt.gz', 'rt') as f:
  for line in f:
    if line.startswith('#'):
      continue
    u, v = map(int, line.split())
    adj.setdefault(u, []).append(v)
    adj.setdefault(v, []) # ensure node exists

start = random.choice(list(adj.keys()))
visited = set([start])
queue = deque([start])

while queue and len(visited) < 300:
  node = queue.popleft()
  for neighbor in adj.get(node, []):
    if neighbor not in visited:
      visited.add(neighbor)
      queue.append(neighbor)
    if len(visited) >= 300:
      break

# keep edges inside sampled nodes
sample_edges = [(u, v) for u in visited for v in adj.get(u, []) if v in visited]

### Preprocessing Note
To handle the large-scale Google Web Graph (which contains over 800k nodes) within a Colab environment, we use a Breadth-First Search (BFS) sampling method starting from a random node. This preserves the local community structure and directional flow while making the computation of complex metrics like betweenness (bridge role) and clustering feasible.

In [ ]:
import pandas as pd

# 1. Rebuild and Enrich Graph
G = nx.DiGraph()
G.add_edges_from(sample_edges)

# Add meaningful attributes: 'role' based on in-degree and 'out_degree' metadata
for node in G.nodes():
    in_d = G.in_degree(node)
    out_d = G.out_degree(node)
    G.nodes[node]['authoritativeness'] = in_d
    G.nodes[node]['out_degree'] = out_d

# 2. Compute Final Metrics
metrics = {}
metrics['Nodes'] = G.number_of_nodes()
metrics['Edges'] = G.number_of_edges()
metrics['Density'] = nx.density(G)

# For clustering and connectivity, we look at the underlying undirected structure
undir_G = G.to_undirected()
metrics['Connected'] = nx.is_connected(undir_G)
metrics['Avg Clustering'] = nx.average_clustering(undir_G)

# Centrality (In-link importance and Bridge role)
in_degree_cent = nx.in_degree_centrality(G)
betweenness_cent = nx.betweenness_centrality(G, normalized=True) # Bridge role

print("--- Final Network Metrics ---")
for k, v in metrics.items():
    print(f"{k}: {v}")

In [ ]:
# 3. Produce Ranked Output (Node Importance Table)
ranking_df = pd.DataFrame({
    'Node_ID': list(G.nodes()),
    'In_Link_Importance': [in_degree_cent[n] for n in G.nodes()],
    'Bridge_Role_Score': [betweenness_cent[n] for n in G.nodes()]
})

# Rank by in-link importance
ranking_df = ranking_df.sort_values('In_Link_Importance', ascending=False).head(15)
print("\n--- Top 15 Pages: In-Link Importance & Bridge Role ---")
display(ranking_df)

In [ ]:
# 4. Final Visualization for Presentation
plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, k=0.3, iterations=50, seed=42)

# Nodes colored by Bridge Role, size by In-Link Importance
node_color = [betweenness_cent[n] for n in G.nodes()]
node_size = [in_degree_cent[n] * 5000 + 50 for n in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_size=node_size, node_color=node_color,
                       cmap=plt.cm.viridis, alpha=0.9)
nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=10, edge_color='gray', alpha=0.15)

plt.title("Google Web Graph: Structural Analysis (BFS Sample)", fontsize=16)
plt.colorbar(plt.cm.ScalarMappable(cmap=plt.cm.viridis), label='Bridge Role (Betweenness)')
plt.axis('off')
plt.savefig("web_graph_final.png", dpi=300)
plt.show()

In [ ]:
# 5. Export Artifacts
ranking_df.to_csv('google_web_ranking.csv', index=False)
nx.write_graphml(G, "google_web_enriched.graphml")

with open('network_metrics.txt', 'w') as f:
    for k, v in metrics.items():
        f.write(f"{k}: {v}\n")

print("Artifacts Exported: google_web_ranking.csv, google_web_enriched.graphml, network_metrics.txt, web_graph_final.png")

### Case-Study Summary: Google Web Graph Topology
This analysis reveals the hierarchical nature of the web. By preserving edge direction, we identified 'Authority' nodes (high In-Link Importance) that serve as major information hubs. Conversely, nodes with high betweenness represent critical 'Bridges' that connect disparate segments of the graph. The low density and presence of these bridges suggest a 'Small-World' architecture where information must pass through specific high-traffic corridors to navigate between different domains.